# Simple LangGraph Ticket Solver (Zero Pipes, Pure Graph)

This notebook solves the **exact same support-ticket problem** as `Simple_LangChain_Ticket_Solver_Core_Only.ipynb`, but using **LangGraph** with **ZERO pipes (`|`) and ZERO Runnables**.

---

### The Power of LangGraph: No Pipe Chains Needed!

In LangGraph:
* **Nodes are just normal Python functions.**
* Inside each node, you simply call `model.invoke()` with regular f-strings.
* **No `ChatPromptTemplate`**, **no `StrOutputParser`**, and **no `|` operators**!
* LangGraph takes care of state sharing, node execution order, and visual graphing.

---

### The Architecture
```text
START
  ↓
1. clean_step       (Node: cleans user whitespace & validates)
  ↓
2. analyse_ticket   (Node: runs 3 direct model.invoke() calls)
  ↓
3. add_decision     (Node: decides if human escalation is needed)
  ↓
4. add_reply        (Node: writes helpful customer reply)
  ↓
END
```


## Google Colab setup

Use a fresh **Python 3.12 or 3.13 CPU runtime** for each notebook. In the Secrets sidebar, add `OPENAI_API_KEY` and enable **Notebook access** for this notebook. Run cells from top to bottom. If installation requests a restart, restart the session and continue with the imports/credentials cell (do not reinstall repeatedly). No Drive mount or local project files are needed.

The default model is `gpt-4o-mini`. API billing/quota and model access are required; a ChatGPT subscription does not supply API quota. Keys are never printed. Package conflicts can occur if multiple framework notebooks share one runtime—start a fresh session when switching frameworks.


## 1. Install Dependencies

Install `langgraph`, `langchain-openai`, and `python-dotenv`. Notice we don't need `langchain-core` prompt parsers!


In [ ]:
%pip -q install "langchain-core==1.6.9" "langchain-openai==1.7.0" "python-dotenv==1.2.4" "langgraph==1.2.14"


## 2. Add your OpenAI API key

The notebook automatically checks in this order:
1. **Local `.env` file** (in project root)
2. **Environment variable** (`OPENAI_API_KEY`)
3. **Google Colab Secrets** (click 🔑 Secrets on the left)
4. **Interactive prompt** (if none of the above are found)


In [ ]:
import os

def read_openai_key():
    # In Colab, always use this notebook's enabled secret.
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError(
                "In Colab Secrets, add OPENAI_API_KEY and enable Notebook access, "
                "then rerun this cell."
            ) from None
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError("The OPENAI_API_KEY Colab secret is empty.")
        return key.strip()

    # Local Python fallback; never overwrite an existing environment key.
    try:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv(usecwd=True), override=False)
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        from getpass import getpass
        key = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not key:
        raise ValueError("OPENAI_API_KEY cannot be empty.")
    return key

os.environ["OPENAI_API_KEY"] = read_openai_key()

print("✅ OpenAI API key is ready.")


## 3. Initialize the Model

Notice: We **only** import `ChatOpenAI`. No prompt templates, no output parsers, and no runnables!


In [ ]:
from langchain_openai import ChatOpenAI

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------

model = ChatOpenAI(
    model="gpt-4o-mini"
)

print("✅ Model ready.")


## 4. Define the LangGraph `State`

In LangGraph, the **State** is the shared whiteboard for the application.

Every node:
1. Receives the current `State` as input.
2. Returns a dictionary with updates to write back to the `State`.


In [ ]:
from typing import TypedDict, Optional

class TicketState(TypedDict):
    ticket: str
    analysis: Optional[dict]
    decision: Optional[str]
    reply: Optional[str]


## 5. Build the Nodes (Pure Python Functions + Direct `model.invoke()`)

Notice how clean each node is:
* Read what you need from `state`.
* Call `model.invoke(f"prompt").content.strip()`.
* Return a dictionary of updates.

**No pipes (`|`) anywhere!**


In [ ]:
# ---------------------------------------------------------
# NODE 1: clean_step
# Problem: user input may contain spaces or may be empty.
# Solution: standard Python logic to clean and validate.
# ---------------------------------------------------------

def clean_node(state: TicketState) -> dict:
    message = state["ticket"].strip()

    if not message:
        raise ValueError("Ticket cannot be empty.")

    return {"ticket": message}


In [ ]:
# ---------------------------------------------------------
# NODE 2: analyse_ticket
# Problem: run summary, category, and urgency analysis.
# Solution: direct model.invoke() calls with regular f-strings!
# ---------------------------------------------------------

def analyse_node(state: TicketState) -> dict:
    ticket = state["ticket"]

    summary = model.invoke(
        f"Summarize this support ticket in one short sentence:\n{ticket}"
    ).content.strip()

    category = model.invoke(
        f"Return only one category: billing, technical, or account.\nTicket: {ticket}"
    ).content.strip()

    urgency = model.invoke(
        f"Return only one urgency: low, medium, or high.\nTicket: {ticket}"
    ).content.strip()

    return {
        "analysis": {
            "summary": summary,
            "category": category,
            "urgency": urgency,
        }
    }


In [ ]:
# ---------------------------------------------------------
# NODE 3: add_decision
# Problem: decide if ticket needs urgent human escalation.
# Solution: simple prompt string + model.invoke().
# ---------------------------------------------------------

def decision_node(state: TicketState) -> dict:
    prompt = f"""
Review this ticket and the initial analysis:

Ticket:
{state['ticket']}

Analysis:
{state['analysis']}

Give the final decision:
1. Should this ticket be escalated to a human? (Yes/No)
2. Brief reason why.
"""

    decision = model.invoke(prompt).content.strip()

    return {"decision": decision}


In [ ]:
# ---------------------------------------------------------
# NODE 4: add_reply
# Problem: write a friendly customer reply.
# Solution: simple prompt string + model.invoke().
# ---------------------------------------------------------

def reply_node(state: TicketState) -> dict:
    prompt = f"""
You are a customer-support assistant.

Customer message:
{state['ticket']}

Decision:
{state['decision']}

Write a short and helpful reply to the customer.
Do not mention prompts, AI, or internal system details.
"""

    reply = model.invoke(prompt).content.strip()

    return {"reply": reply}


## 6. Assemble the Graph (`StateGraph`)

Now we connect our nodes together using LangGraph edges:
1. `builder = StateGraph(TicketState)`
2. `builder.add_node(name, function)`
3. `builder.add_edge(from, to)`
4. `app = builder.compile()`


In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Initialize graph with State
builder = StateGraph(TicketState)

# 2. Add each node (step)
builder.add_node("clean_step", clean_node)
builder.add_node("analyse_ticket", analyse_node)
builder.add_node("add_decision", decision_node)
builder.add_node("add_reply", reply_node)

# 3. Connect nodes with edges (START -> 1 -> 2 -> 3 -> 4 -> END)
builder.add_edge(START, "clean_step")
builder.add_edge("clean_step", "analyse_ticket")
builder.add_edge("analyse_ticket", "add_decision")
builder.add_edge("add_decision", "add_reply")
builder.add_edge("add_reply", END)

# 4. Compile into a runnable application
app = builder.compile()

print("✅ LangGraph ticket workflow compiled successfully!")


## 7. Visualize the Graph

LangGraph can automatically render a visual Mermaid flowchart diagram of your graph!


In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())


## 8. The Runner Function (`solve_ticket`)

One simple function that invokes `app.invoke({"ticket": message})` and prints the result.


In [ ]:
def solve_ticket(message: str):
    """
    Give this function one customer message.
    It runs the LangGraph application and prints the result.
    """

    result = app.invoke({"ticket": message})

    print("\n========== TICKET RESULT (LangGraph) ==========")
    print("Message  :", result["ticket"])
    print("\nAnalysis :")
    for key, val in result["analysis"].items():
        print(f"  - {key}: {val}")
    print("\nDecision :\n", result["decision"])
    print("\nReply to customer:\n", result["reply"])
    print("===============================================")

    return result


print("✅ LangGraph runner function ready.")
print("Now call: solve_ticket('your message')")


## 9. Run ONE function

Test the entire LangGraph flow with a realistic customer message!


In [ ]:
result = solve_ticket(
    "My account is locked and I have an important meeting in 10 minutes."
)


# Summary: Why LangGraph without Pipes is so Clean

### What LangGraph gives you:
1. **Zero pipe complexity**: No `|`, no `RunnableParallel`, no `RunnablePassthrough`.
2. **Standard Python inside nodes**: Just read `state["ticket"]`, call `model.invoke(prompt)`, and return a dictionary.
3. **Explicit State**: `TicketState` is an open blackboard that is easy to inspect and debug.
4. **Visual Flow**: Built-in Mermaid rendering to literally see the architecture diagram.
